## 先统计模型输出的结果数据，进行数据分析

In [1]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import os
from tqdm import tqdm

def calculate_class_similarity(csv_path, class_label=0):


    print(f"读取CSV文件: {csv_path}")
    df = pd.read_csv(csv_path)
    print(f"总样本数: {len(df)}")
    
    # 3. 筛选指定类别的样本
    df_class = df[df['prediction'] == class_label].copy()
    print(f"类别 {class_label} 样本数: {len(df_class)}")
    

    # 4. 计算每个样本与类别平均特征的余弦相似度
    confidences = []
    valid_indices = []
    
    for idx, row in tqdm(df_class.iterrows(), total=len(df_class), desc=f"计算类别{class_label}相似度"):

        confidence = row['confidence']
        confidences.append(confidence)
        valid_indices.append(idx)

    # 5. 将相似度添加到DataFrame
    df_class.loc[valid_indices, 'confidence'] = confidences
    
    # 6. 统计信息
    sim_array = np.array(confidences)
    
    stats = {
        'class': class_label,
        'count': len(sim_array),
        'mean': np.mean(sim_array),
        'std': np.std(sim_array),
        'min': np.min(sim_array),
        'max': np.max(sim_array),
        'median': np.median(sim_array),
        'q1': np.percentile(sim_array, 25),
        'q3': np.percentile(sim_array, 75),
        'iqr': np.percentile(sim_array, 75) - np.percentile(sim_array, 25),
        'p5': np.percentile(sim_array, 5),
        'p95': np.percentile(sim_array, 95),
    }
    
    # 打印统计信息
    print("\n" + "="*50)
    print(f"类别 {class_label} 相似度统计")
    print("="*50)
    print(f"样本数量: {stats['count']}")
    print(f"均值 (mean): {stats['mean']:.6f}")
    print(f"标准差 (std): {stats['std']:.6f}")
    print(f"最小值 (min): {stats['min']:.6f}")
    print(f"最大值 (max): {stats['max']:.6f}")
    print(f"中位数 (median): {stats['median']:.6f}")
    print(f"第一四分位数 (Q1): {stats['q1']:.6f}")
    print(f"第三四分位数 (Q3): {stats['q3']:.6f}")
    print(f"四分位距 (IQR): {stats['iqr']:.6f}")
    print(f"5% 分位数: {stats['p5']:.6f}")
    print(f"95% 分位数: {stats['p95']:.6f}")
    
    return df_class, stats

def calculate_both_classes(csv_path):
    """
    计算类别0和类别1的相似度统计
    """
    print("="*60)
    print("计算两个类别的相似度统计")
    print("="*60)
    
    # 计算类别0
    df_class0, stats0 = calculate_class_similarity(csv_path, class_label=0)
    
    print("\n")
    
    # 计算类别1
    df_class1, stats1 = calculate_class_similarity(csv_path, class_label=1)
    

    return df_class0,df_class1



# 使用示例
if __name__ == "__main__":
    # 文件路径

    csv_path = "/home/user/prognosis_lst/evaluate_model/feature_similiary/lung1/inference_results.csv"
    df_class0,df_class1 = calculate_both_classes(csv_path)

计算两个类别的相似度统计
读取CSV文件: /home/user/prognosis_lst/evaluate_model/feature_similiary/lung1/inference_results.csv
总样本数: 38886
类别 0 样本数: 13969


计算类别0相似度: 100%|██████████| 13969/13969 [00:00<00:00, 56254.00it/s]



类别 0 相似度统计
样本数量: 13969
均值 (mean): 0.871970
标准差 (std): 0.140871
最小值 (min): 0.500153
最大值 (max): 0.999996
中位数 (median): 0.935387
第一四分位数 (Q1): 0.791450
第三四分位数 (Q3): 0.984247
四分位距 (IQR): 0.192797
5% 分位数: 0.566445
95% 分位数: 0.998650


读取CSV文件: /home/user/prognosis_lst/evaluate_model/feature_similiary/lung1/inference_results.csv
总样本数: 38886
类别 1 样本数: 24917


计算类别1相似度: 100%|██████████| 24917/24917 [00:00<00:00, 57610.31it/s]


类别 1 相似度统计
样本数量: 24917
均值 (mean): 0.921926
标准差 (std): 0.123085
最小值 (min): 0.500065
最大值 (max): 1.000000
中位数 (median): 0.984776
第一四分位数 (Q1): 0.903275
第三四分位数 (Q3): 0.998129
四分位距 (IQR): 0.094854
5% 分位数: 0.611211
95% 分位数: 0.999899


## 保留模型输出和距离筛选一致的数据

In [3]:
import pandas as pd

# 一行搞定：读取、合并、筛选
df = pd.merge(
    pd.read_csv('/home/user/prognosis_lst/evaluate_model/feature_similiary/lung1/inference_results.csv'),
    pd.read_csv('/home/user/prognosis_lst/evaluate_model/feature_similiary/lung1/pseudo_label_results.csv'),
    on='feature_path'
).query('label == prediction')
print(len(df))
# 保存
df.to_csv('/home/user/prognosis_lst/evaluate_model/feature_similiary/lung1/inference_distance_samples.csv', index=False)
print(f"找到 {len(df)} 个一致样本")

34566
找到 34566 个一致样本


## 再用置信度筛选一下,而且要做到类别均衡

In [5]:
import pandas as pd

# 读取数据
df = pd.read_csv('/home/user/prognosis_lst/evaluate_model/feature_similiary/lung1/inference_distance_samples.csv')
print(f"原始数据总样本数: {len(df)}")
print("\n原始数据各类别数量:")
print(df['label'].value_counts())

# 根据不同类别设置不同的筛选条件
# 假设：类别0用阈值A，类别1用阈值B
threshold_class0 = 0.871970  # 类别0的置信度阈值
threshold_class1 = 0.921926  # 类别1的置信度阈值

# 按类别分别筛选
df_class0 = df[(df['label'] == 0) & (df['confidence'] > threshold_class0)]
df_class1 = df[(df['label'] == 1) & (df['confidence'] > threshold_class1)]

# 合并筛选结果
df_filtered = pd.concat([df_class0, df_class1], ignore_index=True)

columns_to_drop = ['confidence', 'prediction']  # 替换为你要删除的实际列名
df_filtered = df_filtered.drop(columns=columns_to_drop, errors='ignore')

# 统计结果
print("\n" + "="*50)
print("筛选结果统计")
print("="*50)
print(f"类别0 (阈值 > {threshold_class0}): {len(df_class0)} 个样本")
print(f"类别1 (阈值 > {threshold_class1}): {len(df_class1)} 个样本")
print(f"总计: {len(df_filtered)} 个样本")

# # 保存筛选后的数据
df_filtered.to_csv('inference_distance_samples_filtered.csv', index=False)
print(f"\n筛选结果已保存到: inference_distance_samples_filtered.csv")

# # 可选：查看筛选前后的对比
# print("\n筛选前后对比:")
# print(f"  原始数据: {len(df)} 个样本")

原始数据总样本数: 34566

原始数据各类别数量:
label
1    22167
0    12399
Name: count, dtype: int64

筛选结果统计
类别0 (阈值 > 0.87197): 8523 个样本
类别1 (阈值 > 0.921926): 17654 个样本
总计: 26177 个样本

筛选结果已保存到: inference_distance_samples_filtered.csv
